In [3]:
import sys
import os
sys.path += [ f'{os.environ["HOME"]}/.local/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages' ]

# Now we can safely import atlasopenmagic
import atlasopenmagic as atom

In [4]:
import uproot # for reading .root files
import time # to measure time to analyse
import math # for mathematical functions such as square root
import awkward as ak # for handling complex and nested data structures efficiently
import numpy as np # # for numerical calculations such as histogramming
import matplotlib.pyplot as plt # for plotting
from matplotlib.ticker import MaxNLocator,AutoMinorLocator # for minor ticks
from lmfit.models import PolynomialModel, GaussianModel # for the signal and background fits
import vector #to use vectors
import requests # for HTTP access
import aiohttp # HTTP client support
import pandas as pd

In [5]:
atom.set_release('2025e-13tev-beta')

Fetching metadata for release: 2025e-13tev-beta...
Fetching datasets: 100%|██████████| 374/374 [00:00<00:00, 1109.38datasets/s]
✓ Successfully cached 374 datasets.
Active release: 2025e-13tev-beta. (Datasets path: REMOTE)


In [6]:
lumi = 36

In [7]:
def get_xsec_weight(metadata, lumi):
    return (
        lumi * 1000
        * metadata["cross_section_pb"]
        * metadata["genFiltEff"]
        * metadata["kFactor"]
        / metadata["sumOfWeights"]
    )

def get_N_inclusive(metadata, lumi):
    return (
        lumi * 1000
        * metadata["cross_section_pb"]
        * metadata["genFiltEff"]
        * metadata["kFactor"]
    )

def get_inclusive_yield(metadata, lumi):
    return (
        lumi * 1000
        * metadata["cross_section_pb"]
        * metadata["genFiltEff"]
        * metadata["kFactor"]
    )

def calc_weight(xsec_weight, weight_arr, data):
    for variable in weight_arr:
        xsec_weight = xsec_weight * data[variable]
    return xsec_weight

In [8]:
variable_weights_arr = ["mcWeight"]
variables = [] + variable_weights_arr

In [9]:
# Loop over all the files in our list
def check_weights(dsid_list, fraction = 1):
    for dsid in dsid_list:
        sum_of_mc_weights = 0
        N_gen_rel = 0
        N_inclusive = 0

        metadata = atom.get_metadata(dsid)
        xsec_weight = get_xsec_weight(metadata, lumi)
        N_inclusive = get_N_inclusive(metadata, lumi)

        file_list = atom.get_urls(dsid, protocol='root', cache=False)
        print("filelist: ", file_list)

        #for url in atom.get_urls(dsid, protocol='root', cache=True):
        #    print("url: ", url)
        for afile in file_list:
                # Print which sample is being processed
            print(f'Processing file {afile} ({file_list.index(afile)+1}/{len(file_list)})')
            print("afile: ", f'{afile}')

            # Open file
            tree = uproot.open(afile + ":analysis")

            numevents = tree.num_entries

            # Perform the cuts for each data entry in the tree and calculate the invariant mass
            for data in tree.iterate(variables, library="ak", entry_stop=numevents*fraction):
                #print("data: ", data)

                sum_of_mc_weights = sum_of_mc_weights + ak.sum(data["mcWeight"])
                print("weights: ", data["mcWeight"])

        N_gen_rel = xsec_weight * sum_of_mc_weights
        print('Done processing data of dsid ', dsid)
        print("sum_of_mc_weights: ", sum_of_mc_weights)
        print("N_gen_rel: ", N_gen_rel)
        print("N_inclusive: ", N_inclusive)
    return sum_of_mc_weights

In [10]:
dsid_list = [301209, 700323, 700324, 700325]

In [13]:
check_weights(dsid_list)

filelist:  ['root://eospublic.cern.ch:1094//eos/opendata/atlas/rucio/user/egramsta/mc_301209.Pythia8EvtGen_A14MSTW2008LO_Zprime_NoInt_mumu_SSM3000.noskim.root']
Processing file root://eospublic.cern.ch:1094//eos/opendata/atlas/rucio/user/egramsta/mc_301209.Pythia8EvtGen_A14MSTW2008LO_Zprime_NoInt_mumu_SSM3000.noskim.root (1/1)
afile:  root://eospublic.cern.ch:1094//eos/opendata/atlas/rucio/user/egramsta/mc_301209.Pythia8EvtGen_A14MSTW2008LO_Zprime_NoInt_mumu_SSM3000.noskim.root
weights:  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..., 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Done processing data of dsid  301209
sum_of_mc_weights:  19928.0
N_gen_rel:  63.555172
N_inclusive:  63.784800000000004
filelist:  ['root://eospublic.cern.ch:1094//eos/opendata/atlas/rucio/user/egramsta/mc_700323.Sh_2211_Zmumu_maxHTpTV2_BFilter.noskim.root']
Processing file root://eospublic.cern.ch:1094//eos/opendata/atlas/rucio/user/egramsta/mc_700323.Sh_2211_Zmumu_maxHTpTV2_BFilter.noskim.root (1/1)
afile:  root://eospub

np.float32(1.1908601e+15)